# Financial Fraud Detection Using Machine Learning
**Python | EDA | Logistic Regression | Random Forest | XGBoost**

## Project objective
Build a classification workflow to identify potentially fraudulent card transactions. The notebook covers data quality, exploratory analysis, class imbalance, model comparison, threshold tuning, and investigation of false positives and false negatives.

**Dataset:** Kaggle *Credit Card Fraud Detection* (`creditcard.csv`)  
Expected target column: `Class` (1 = fraud, 0 = legitimate).

> Important: This is an educational portfolio project, not a production fraud decision system. Do not use the test set for model selection. Fraud labels are extremely imbalanced, so accuracy alone is misleading.

## 1. Import libraries and configure the environment
Run this notebook in Kaggle. If XGBoost is not available in the environment, install it in a Kaggle notebook with internet enabled, or use the fallback note in the model section.

In [ ]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception as e:
    XGBOOST_AVAILABLE = False
    print("XGBoost import unavailable:", e)

## 2. Load the Kaggle dataset

Attach the Kaggle dataset to this notebook using **Add Input**. The loader searches `/kaggle/input` for `creditcard.csv`. If you use a different filename, update `DATA_PATH`.

In [ ]:
DATA_PATH = None  # Example: "/kaggle/input/creditcardfraud/creditcard.csv"

if DATA_PATH is None:
    candidates = list(Path("/kaggle/input").rglob("creditcard.csv")) if Path("/kaggle/input").exists() else []
    if not candidates:
        raise FileNotFoundError(
            "Could not find creditcard.csv. Add the Credit Card Fraud Detection dataset "
            "through Kaggle's Add Input button, or set DATA_PATH manually."
        )
    DATA_PATH = str(candidates[0])

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 3. Data quality checks

In [ ]:
display(df.info())
display(df.isna().sum().sort_values(ascending=False).head(15))
print("Duplicate rows:", df.duplicated().sum())
print("Target distribution:")
display(df["Class"].value_counts(dropna=False).rename("count").to_frame())

In [ ]:
if "Class" not in df.columns:
    raise ValueError("Expected target column 'Class' was not found.")

df = df.drop_duplicates().copy()
df = df.dropna(subset=["Class"])
df["Class"] = df["Class"].astype(int)

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Class")
plt.title("Legitimate vs Fraudulent Transactions")
plt.xlabel("Class (0 = legitimate, 1 = fraud)")
plt.ylabel("Number of transactions")
plt.show()

fraud_rate = df["Class"].mean() * 100
print(f"Fraud share: {fraud_rate:.4f}%")

## 4. Exploratory data analysis
The dataset commonly contains `Time`, `Amount`, anonymized PCA features (`V1`–`V28`) and `Class`. Examine the transaction amount and time distributions without assuming that any one feature alone proves fraud.

In [ ]:
if "Amount" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    sns.histplot(data=df, x="Amount", hue="Class", bins=50, element="step",
                 stat="density", common_norm=False, ax=axes[0])
    axes[0].set_xlim(0, df["Amount"].quantile(0.99))
    axes[0].set_title("Transaction amount (up to 99th percentile)")
    sns.boxplot(data=df, x="Class", y="Amount", showfliers=False, ax=axes[1])
    axes[1].set_title("Amount by transaction class")
    plt.tight_layout()
    plt.show()

if "Time" in df.columns:
    df["Hour_Approx"] = ((df["Time"] // 3600) % 24).astype(int)
    plt.figure(figsize=(10, 4))
    sns.histplot(data=df, x="Hour_Approx", hue="Class", bins=24,
                 multiple="layer", stat="density", common_norm=False)
    plt.title("Transaction time pattern (approximate hour)")
    plt.show()

## 5. Prepare features and split the data
Use a stratified split so that the very small fraud class is represented in both sets. Keep the test set untouched until final evaluation.

In [ ]:
target = "Class"
drop_cols = [target]
X = df.drop(columns=drop_cols).copy()
y = df[target].copy()

# Remove helper column only if it was created for EDA; it is derived from Time,
# so retaining it is optional. Keep it here as a candidate feature.
X = X.replace([np.inf, -np.inf], np.nan)
for col in X.select_dtypes(include=np.number).columns:
    X[col] = X[col].fillna(X[col].median())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Train fraud rate:", round(y_train.mean()*100, 4), "%")
print("Test fraud rate:", round(y_test.mean()*100, 4), "%")

## 6. Train classification models
Logistic Regression uses scaling. Random Forest and XGBoost can work without standardization. `class_weight` / `scale_pos_weight` help address imbalance, but model thresholds still need business-led tuning.

In [ ]:
models = {}

lr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ))
])
models["Logistic Regression"] = lr

rf = RandomForestClassifier(
    n_estimators=300, max_depth=None, min_samples_leaf=2,
    class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE
)
models["Random Forest"] = rf

if XGBOOST_AVAILABLE:
    neg, pos = np.bincount(y_train.astype(int))
    xgb = XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        scale_pos_weight=neg / max(pos, 1),
        eval_metric="aucpr", random_state=RANDOM_STATE,
        n_jobs=-1
    )
    models["XGBoost"] = xgb

for name, model in models.items():
    print("Training", name)
    model.fit(X_train, y_train)

## 7. Compare models
For an imbalanced fraud dataset, prioritize **PR-AUC, recall and precision** alongside ROC-AUC. The appropriate trade-off depends on investigation capacity and the cost of missed fraud.

In [ ]:
results = []
probabilities = {}

for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    probabilities[name] = proba
    pred_05 = (proba >= 0.5).astype(int)
    results.append({
        "Model": name,
        "Precision@0.5": precision_score(y_test, pred_05, zero_division=0),
        "Recall@0.5": recall_score(y_test, pred_05, zero_division=0),
        "F1@0.5": f1_score(y_test, pred_05, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, proba),
        "PR_AUC": average_precision_score(y_test, proba)
    })

comparison = pd.DataFrame(results).sort_values("PR_AUC", ascending=False)
display(comparison)

In [ ]:
plt.figure(figsize=(8, 5))
for name, proba in probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test, proba)
    plt.plot(recall, precision, label=name)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curves")
plt.legend()
plt.show()

## 8. Select a threshold using validation data
A default threshold of 0.5 is not automatically suitable. This example uses a validation split from the training data and chooses the threshold with the highest F1 score. In a real bank, the threshold should reflect fraud-loss costs, customer friction, and analyst capacity.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

# Demonstration threshold selection using the best PR-AUC model.
best_name = comparison.iloc[0]["Model"]
best_model = models[best_name]
best_model.fit(X_fit, y_fit)
val_proba = best_model.predict_proba(X_val)[:, 1]

p, r, thresholds = precision_recall_curve(y_val, val_proba)
f1_values = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
best_idx = int(np.argmax(f1_values))
chosen_threshold = float(thresholds[best_idx])

print("Selected model:", best_name)
print("Validation threshold (max F1):", round(chosen_threshold, 4))
print("Validation precision:", round(p[best_idx], 4))
print("Validation recall:", round(r[best_idx], 4))

## 9. Final test evaluation and confusion matrix
The threshold was selected using validation data. Evaluate once on the held-out test set.

In [ ]:
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= chosen_threshold).astype(int)

print(classification_report(y_test, test_pred, target_names=["Legitimate", "Fraud"], zero_division=0))
cm = confusion_matrix(y_test, test_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted legitimate", "Predicted fraud"],
            yticklabels=["Actual legitimate", "Actual fraud"])
plt.title(f"Confusion Matrix — {best_name}")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

tn, fp, fn, tp = cm.ravel()
print({"True negatives": tn, "False positives": fp,
       "False negatives": fn, "True positives": tp})

## 10. Investigate false positives and false negatives
False positives are legitimate transactions flagged for review. False negatives are fraud cases the model missed. Review examples to understand patterns; do not assume a model explanation establishes causality.

In [ ]:
error_analysis = X_test.copy()
error_analysis["Actual_Class"] = y_test.values
error_analysis["Fraud_Probability"] = test_proba
error_analysis["Predicted_Class"] = test_pred
error_analysis["Error_Type"] = np.select(
    [
        (error_analysis["Actual_Class"] == 0) & (error_analysis["Predicted_Class"] == 1),
        (error_analysis["Actual_Class"] == 1) & (error_analysis["Predicted_Class"] == 0)
    ],
    ["False Positive", "False Negative"],
    default="Correct"
)

display(error_analysis[error_analysis["Error_Type"] == "False Positive"].head(10))
display(error_analysis[error_analysis["Error_Type"] == "False Negative"].head(10))
display(error_analysis["Error_Type"].value_counts().to_frame("Transactions"))

## 11. Feature importance
Tree-based feature importance is a useful first diagnostic, but it can be biased and does not prove causation. For a stronger portfolio extension, add SHAP explanations and assess stability across time periods.

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importance = pd.Series(best_model.feature_importances_, index=X_train.columns)
    importance = importance.sort_values(ascending=False).head(15)
    plt.figure(figsize=(9, 5))
    sns.barplot(x=importance.values, y=importance.index)
    plt.title(f"Top feature importances — {best_name}")
    plt.xlabel("Importance")
    plt.show()
elif best_name == "XGBoost":
    importance = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)
    display(importance.to_frame("importance"))
else:
    print("For Logistic Regression, inspect standardized coefficients separately.")

## 12. Business impact illustration
Use transparent assumptions rather than presenting estimates as actual bank savings. Replace these assumptions with business-approved values when available.

In [ ]:
# Illustrative assumptions only — edit these for a scenario analysis.
average_fraud_loss = 50000      # currency units per missed fraudulent transaction
review_cost = 150              # cost to investigate one flagged transaction

missed_fraud_loss = fn * average_fraud_loss
investigation_cost = (tp + fp) * review_cost

print(f"Illustrative missed-fraud loss: {missed_fraud_loss:,.0f}")
print(f"Illustrative investigation cost: {investigation_cost:,.0f}")
print("These figures are scenario assumptions, not observed dataset financial losses.")

## 13. Export predictions for Power BI
Export a transaction-level file with actual label, predicted probability, predicted class, and error type. In a real deployment, avoid exporting sensitive customer information.

In [ ]:
powerbi_output = X_test.copy()
powerbi_output["Actual_Fraud"] = y_test.values
powerbi_output["Fraud_Probability"] = test_proba
powerbi_output["Predicted_Fraud"] = test_pred
powerbi_output["Prediction_Outcome"] = error_analysis["Error_Type"].values

powerbi_output.to_csv("/kaggle/working/fraud_model_predictions.csv", index=False)
comparison.to_csv("/kaggle/working/fraud_model_comparison.csv", index=False)
print("Saved files in /kaggle/working/")

## 14. Conclusion and portfolio discussion
Complete this section after running the notebook. Record:
1. Best model based on PR-AUC and business constraints.
2. Precision and recall at the selected threshold.
3. Number of false positives and false negatives.
4. Which features were most influential.
5. Limitations: anonymized features, historical data, potential concept drift, and no production monitoring.

### Suggested Power BI pages
- **Executive Overview:** transactions, fraud count, fraud rate, estimated exposure.
- **Fraud Trends:** transaction time and amount patterns.
- **Model Performance:** precision, recall, PR-AUC, confusion matrix.
- **Investigation Queue:** high-probability transactions and false-positive review.

### Suggested interview talking point
"Because fraud is a highly imbalanced classification problem, I did not rely on accuracy. I compared PR-AUC, precision and recall, selected a threshold using validation data, and separately investigated false positives and false negatives to understand the operational trade-off."